# 1. Experiment Information
This notebook provides the standard execution pipeline for baseline two-stem (vocals/accompaniment) source separation experiments using the pretrained HT-Demucs model on Indian Folk Music datasets.

# 2. Google Drive Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Repository Configuration

In [4]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path("/content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation")
print(f"Project Root: {PROJECT_ROOT}")

Project Root: /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation


# 4. Experiment Configuration

In [5]:
# --- EXPERIMENT CONFIGURATION ---

EXPERIMENT_ID = "EXP001"
NOTEBOOK_VERSION = "1.0"
MODEL_NAME = "HT-Demucs"
DATASET_NAME = "BIHU"
INPUT_AUDIO_FILENAME = "BIHU_001.wav"
TWO_STEMS = "vocals"

# Derived Paths
DATASET_DIR = PROJECT_ROOT / "03_Datasets"
TRADITION_DIR = DATASET_DIR / DATASET_NAME
INPUT_AUDIO_PATH = TRADITION_DIR / "Audio" / "WAV" / INPUT_AUDIO_FILENAME

EXPERIMENT_DIR = PROJECT_ROOT / "05_Experiments" / "Baseline_HT_Demucs" / EXPERIMENT_ID
OUTPUT_DIRECTORY = EXPERIMENT_DIR / "outputs"
LOG_DIRECTORY = EXPERIMENT_DIR / "logs"

print("Configuration Loaded:")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Input Audio  : {INPUT_AUDIO_PATH}")
print(f"Output Dir   : {OUTPUT_DIRECTORY}")
print(f"Log Dir      : {LOG_DIRECTORY}")

Configuration Loaded:
Experiment ID: EXP001
Input Audio  : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/BIHU/Audio/WAV/BIHU_001.wav
Output Dir   : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP001/outputs
Log Dir      : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP001/logs


# 5. Environment Validation

In [6]:
import torch
import sys

# 1. Check Paths
required_paths = {
    "Project Directory": PROJECT_ROOT,
    "Dataset Directory": DATASET_DIR,
    "Input Audio": INPUT_AUDIO_PATH,
    "Output Directory": OUTPUT_DIRECTORY,
    "Log Directory": LOG_DIRECTORY
}

for name, path in required_paths.items():
    if not path.exists():
        raise RuntimeError(f"❌ ERROR: {name} does not exist at {path}")
    else:
        print(f"✓ {name} found.")

# 2. Check CUDA & PyTorch
print(f"✓ PyTorch Version: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("❌ ERROR: CUDA is not available. Please enable GPU in Colab.")
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✓ CUDA Available (GPU: {gpu_name})")

# 3. Check Demucs
try:
    import demucs
    print(f"✓ Demucs installed (Version: {demucs.__version__})")
    demucs_version = demucs.__version__
except ImportError:
    print("Demucs not found. Installing now...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "demucs"], check=True)
    import demucs
    print(f"✓ Demucs installed (Version: {demucs.__version__})")
    demucs_version = demucs.__version__

print("\nEnvironment validation passed!")

✓ Project Directory found.
✓ Dataset Directory found.
✓ Input Audio found.
✓ Output Directory found.
✓ Log Directory found.
✓ PyTorch Version: 2.11.0+cu128
✓ CUDA Available (GPU: Tesla T4)
Demucs not found. Installing now...
✓ Demucs installed (Version: 4.1.0)

Environment validation passed!


# 6. HT-Demucs Execution

In [7]:
import time
import subprocess
import datetime

print(f"Starting {MODEL_NAME} separation on {INPUT_AUDIO_FILENAME}...")

command = [
    sys.executable, "-m", "demucs",
    "--two-stems", TWO_STEMS,
    "-d", "cuda",
    "-o", str(OUTPUT_DIRECTORY),
    str(INPUT_AUDIO_PATH)
]

print("Command:", " ".join(command))

start_time = time.time()
start_dt = datetime.datetime.now()

# Execute Demucs
result = subprocess.run(command)

end_time = time.time()
end_dt = datetime.datetime.now()

if result.returncode != 0:
    raise RuntimeError("❌ ERROR: Demucs execution failed.")
else:
    print("✓ Demucs execution completed.")

processing_duration = end_time - start_time
print(f"Processing took {processing_duration:.2f} seconds.")

Starting HT-Demucs separation on BIHU_001.wav...
Command: /usr/bin/python3 -m demucs --two-stems vocals -d cuda -o /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP001/outputs /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/BIHU/Audio/WAV/BIHU_001.wav
✓ Demucs execution completed.
Processing took 31.61 seconds.


# 7. Output Processing

In [8]:
import os
import shutil

# Demucs creates outputs in OUTPUT_DIRECTORY / htdemucs / <input_stem>
demucs_output_dir = OUTPUT_DIRECTORY / "htdemucs" / INPUT_AUDIO_PATH.stem

vocals_original = demucs_output_dir / "vocals.wav"
no_vocals_original = demucs_output_dir / "no_vocals.wav"

input_stem = INPUT_AUDIO_PATH.stem
vocals_new = demucs_output_dir / f"{input_stem}_vocals.wav"
no_vocals_new = demucs_output_dir / f"{input_stem}_no_vocals.wav"

if vocals_original.exists():
    vocals_original.rename(vocals_new)
    print(f"Renamed: {vocals_original.name} -> {vocals_new.name}")

if no_vocals_original.exists():
    no_vocals_original.rename(no_vocals_new)
    print(f"Renamed: {no_vocals_original.name} -> {no_vocals_new.name}")

Renamed: vocals.wav -> BIHU_001_vocals.wav
Renamed: no_vocals.wav -> BIHU_001_no_vocals.wav


# 8. Execution Log Generation

In [9]:
log_file_path = LOG_DIRECTORY / "execution_log.txt"

command_str = " ".join(command)

log_content = f"""Experiment ID: {EXPERIMENT_ID}
Notebook Version: {NOTEBOOK_VERSION}
Execution Date: {start_dt.strftime('%Y-%m-%d')}
Execution Time: {start_dt.strftime('%H:%M:%S')} - {end_dt.strftime('%H:%M:%S')}
Processing Duration: {processing_duration:.2f} seconds
Model: {MODEL_NAME}
Dataset: {DATASET_NAME}
Input Audio: {INPUT_AUDIO_FILENAME}
Output Directory: {OUTPUT_DIRECTORY}
GPU Name: {gpu_name}
PyTorch Version: {torch.__version__}
Demucs Version: {demucs_version}
Execution Command: {command_str}
Status: Success
Generated Output Files:
- {vocals_new.name}
- {no_vocals_new.name}
"""

with open(log_file_path, "w", encoding="utf-8") as f:
    f.write(log_content)

print(f"Execution log saved to {log_file_path}")

Execution log saved to /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP001/logs/execution_log.txt


# 9. Result Summary

In [10]:
import datetime

print("====================================")
print("Experiment Completed Successfully")
print(f"Experiment:\n{EXPERIMENT_ID}\n")
print(f"Input:\n{INPUT_AUDIO_FILENAME}\n")
print("Generated Files:")
print(f"✓ {vocals_new.name}")
print(f"✓ {no_vocals_new.name}\n")
print(f"Execution Log:\n✓ execution_log.txt\n")
print(f"Processing Time:\n{datetime.timedelta(seconds=int(processing_duration))}")
print("====================================")

Experiment Completed Successfully
Experiment:
EXP001

Input:
BIHU_001.wav

Generated Files:
✓ BIHU_001_vocals.wav
✓ BIHU_001_no_vocals.wav

Execution Log:
✓ execution_log.txt

Processing Time:
0:00:31
